<img src="../logo.png" alt="Vegeta" width="240">

# Perseus — a tracked rover that cuts tree branches

**Perseus** is a ~180 kg tracked machine for cutting branches: the Pekari Rover's track architecture (notebook 30:
rear drive sprocket, front idler, road wheels on bogies, a belt of hinged links) scaled by its parameters, and the
Onager Manus's arm pattern (notebook 22) as one boom on the roof that carries a 2 kW battery chainsaw. The boom has
four axes: a pedestal (yaw), the **boom** and the **stick**, each moved by an electric cylinder (ball screw with a
brake) as on a small excavator, and the **saw head** (wrist pitch) with the saw body and a 40 cm bar.

```
CAD (Dedalus): hull, tracks, pedestal, boom, stick, saw head, cylinders ─► mass budget, CG with the boom stowed
masses ─► ground pressure, MMP ─► Bekker sinkage and compaction per soil ─► resistance vs the drives, grades, range
boom ─► workspace ─► joint torques with the saw and the cutting load vs the drives (cylinder levers vary with pose)
saw + wood ─► cut time, chain and feed forces, head loads, pinch or open, energy per cut, cuts per charge
boom poses × cutting load ─► tip-over margins, track and road-wheel loads
boom tube ─► FEA under the head loads (Talos), modes against the shoulder servo
MuJoCo (Chiron): the machine on rough ground with a side bank, a tree with three welded branches ─► the job
```

The design modules: `designs/perseus_rover.py` (CAD), `designs/perseus_robot.py` (numbers and the MuJoCo model),
`designs/cutting.py` (the saw), `designs/perseus_controller.py` and `designs/perseus_scenario.py` (the job).

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos
from vegeta.dedalus import viz as dviz
sys.path.insert(0, "../designs")
import actuators as act
import cutting, tracks, terramechanics as tm
import pekari_rover_robot as prr
import perseus_robot as pbr
from perseus_rover import PerseusRover, PARTS

RUNS = Path("../_runs/perseus_rover"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
OUT = Path("../output/37_perseus_rover"); OUT.mkdir(parents=True, exist_ok=True)
for f in ("pekari_rover.py", "perseus_rover.py"):           # the Perseus class builds on the Pekari's
    shutil.copy(Path("../designs") / f, RUNS / f)
sys.path.insert(0, str(RUNS))
design = dedalus.load_design(f"{RUNS / 'perseus_rover.py'}:PerseusRover")
V = pbr.PERSEUS
G = 9.81
SAW = cutting.get_saw(V.saw)
SPEC = pd.Series({"mass [kg]": "≤ 200", "branches": "to 120 mm (40 cm bar)", "reach, shoulder to bar tip [m]": "≥ 2.3",
                  "climb, dry ground": "30°", "ground pressure [kPa]": "< 15 (a person on foot: 15-20)",
                  "work time on one charge [h]": "≥ 2", "steering": "skid, pivot turn"}, name="Perseus — targets")
display(SPEC.to_frame())
pd.DataFrame(design.params.table()).set_index("name")

In [ ]:
from vegeta import yanus
GATE = yanus.enter("37_perseus_rover", debug=DEBUG)              # the door keeper (workflows/flow.py): needs; GateClosed names every fix


## 1. The machine

The CAD builds the whole machine and each part on its own. The mass budget takes the hull's sheet from the CAD
area, the links, sprockets and wheels from the CAD volumes (the wheels ribbed to 35 % of solid), the boom and stick
tubes from their CAD volumes, the drives and cylinders from the catalogue, the saw from `cutting.SAWS` and the rest
from a parts list. The battery goes where it balances the machine, boom stowed, over the road wheels' centre.

In [ ]:
p = design.resolve()
parts = {k: design.generate(part=k) for k in PARTS}
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.2) for k, g in parts.items()}
display(pd.DataFrame({k: {"volume_cm3": g.measure()["volume"] / 1e3, "area_cm2": g.measure()["surface_area"] / 1e2,
                          "size_mm": " × ".join(f"{d:.0f}" for d in g.measure()["dimensions"])}
                      for k, g in parts.items()}).T)
dviz.show(dviz.plot3d(parts["rover"]))

In [ ]:
# the stored CAD numbers (perseus_robot.PERSEUS_CAD) against this build: they drive the mass budget
fresh = pbr.cad_numbers(recompute=True)
drift = {k: abs(fresh[k] - v) / v for k, v in V.cad.items() if not isinstance(v, tuple) and v}
assert all(d < 0.02 for d in drift.values()), drift
budget = pd.Series(pbr.mass_budget(), name="kg")
arm = pd.Series(pbr.arm_masses(), name="kg")
cg_stow = pbr.cg()
g = prr.geometry(pbr.chassis_params())
ag = pbr.arm_geometry()
display(budget.round(2).to_frame()); display(arm.round(2).to_frame())
print(f"total {budget.sum():.1f} kg, of which the boom assembly {arm.sum():.1f} kg; CG stowed x {cg_stow['x'] * 1e3:.0f} mm "
      f"(from the road wheels' centre), z {cg_stow['z'] * 1e3:.0f} mm; battery at x {cg_stow['battery_x'] * 1e3:.0f} mm")
print(f"track: b {g['b'] * 1e3:.0f} mm, contact L {g['L'] * 1e3:.0f} mm, gauge B {g['B'] * 1e3:.0f} mm, {g['links']} links per side; "
      f"reach (shoulder to bar tip) {ag['reach']:.2f} m, shoulder {ag['z_world']:.2f} m above the ground")

In [ ]:
# side view: the belt around the sprocket, idler and road wheels, the hull, and the boom in three poses
import pekari_rover as pk
fig, ax = plt.subplots(figsize=(13, 5))
poly = np.array(pk._hull_of_circles([(x * 1e3, z * 1e3, r * 1e3) for x, z, r in g["belt_circles"]], 180))
ax.fill(poly[:, 0], poly[:, 1], color="#2b2b2b", alpha=0.2, label="track belt (pin line)")
Lh, Hh, z0 = p["hull_length"], p["hull_height"], p["ground_clearance"]
ax.add_patch(plt.Rectangle((-Lh / 2, z0), Lh, Hh, color="#d9a21b", alpha=0.5, label="hull"))
for (b_, s_, h_), c, lab in (((V.stow["boom"], V.stow["stick"], V.stow["head"]), "#555", "stowed"),
                             ((p["boom_deg"], p["stick_deg"], p["head_deg"]), "#c06000", "ready"),
                             ((10.0, -10.0, 0.0), "#2a6fb0", "reaching out")):
    pts = PerseusRover.arm_points(p, b_, s_, h_)
    xs, zs = zip(*[pts[k] for k in ("shoulder", "elbow", "wrist", "bar_root", "bar_tip")])
    ax.plot(xs, zs, "-o", color=c, lw=2.5, ms=4, label=f"boom {lab}")
c = pbr.cg()
ax.plot(c["x"] * 1e3, c["z"] * 1e3, "kx", ms=12, mew=3, label="CG, stowed")
ax.axhline(0, color="#8d7b5e"); ax.set(aspect="equal", xlabel="x [mm]", ylabel="z [mm]", title="Perseus, side view")
ax.legend(loc="upper left", fontsize=8); plt.show()

## 2. Ground and drive

The nominal ground pressure W / (2 b L) and Rowland's mean maximum pressure (MMP, the peaks under the road wheels);
Bekker's law turns pressure into sinkage and the rut into compaction resistance. The drive: each track has a
`perseus track drive 1.5 kW, 48 V` on its linear torque–speed line, against the resistance (compaction or rolling,
the track's internal loss f_in = 0.035 + 0.002 v — Pekari's assumption, to be measured — the grade) and the grip
the ground gives.

In [ ]:
W = budget.sum() * G
p_nom = tracks.ground_pressure(W, g["b"], g["L"])
mmp = tm.mmp_rowland(W, g["n_wheels"], g["b"], g["d_wheel"], g["pitch"])
print(f"{W:.0f} N: nominal {p_nom / 1e3:.1f} kPa, MMP {mmp / 1e3:.1f} kPa")
soil_rows = {}
for name, s in tm.SOILS.items():
    soil_rows[name] = {"sinkage [mm]": 0.0 if s.rigid else tm.track_sinkage(W / 2, g["b"], g["L"], s) * 1e3,
                       "compaction [% W]": 0.0 if s.rigid else 2 * tm.compaction_resistance_track(W / 2, g["b"], g["L"], s) / W * 100,
                       "max thrust [% W]": 2 * tm.max_thrust_track(W / 2, g["b"], g["L"], s) / W * 100}
soil_tab = pd.DataFrame(soil_rows).T
display(soil_tab.round(1))

In [ ]:
drive_rows = {}
for soil in ("asphalt", "gravel", "grass", "sandy loam", "dry sand", "mud"):
    for grade in (0.0, 15.0, 30.0):
        d = pbr.drive_check(soil=soil, grade_deg=grade, v_ms=0.5)
        drive_rows[(soil, grade)] = {"resistance [N]": d["resistance_N"], "drives, rated [N]": d["drive_rated_N"],
                                     "drives, stall at 0.5 m/s [N]": d["drive_stall_N"], "traction [N]": d["traction_N"],
                                     "rated holds": d["ok_rated"], "grip holds": d["ok_traction"]}
drive_tab = pd.DataFrame(drive_rows).T
drive_tab.index.names = ["soil", "grade [deg]"]
display(drive_tab.round(0))
d0 = pbr.drive_check(soil="gravel", grade_deg=0.0, v_ms=1.0)
P_drive = d0["resistance_N"] * 1.0 / 0.85                      # 1 m/s on gravel, motor and gearbox 0.85
work_W = SAW.power_W * 0.35 + 150.0                            # a working hour: the saw a third of the time, the boom ~150 W
end_tab = pd.Series({"top speed on the flat [m/s]": d0["v_top_flat_m_s"],
                     "driving 1 m/s on gravel [W]": P_drive, "range at 1 m/s on gravel [km]": V.battery_Wh * 0.8 / (P_drive + V.electronics_W) * 3.6,
                     "cutting work, mean [W]": work_W, "work time on one charge [h]": V.battery_Wh * 0.8 / (work_W + V.electronics_W)},
                    name="endurance (80 % of the battery)")
display(end_tab.round(2).to_frame())

## 3. The boom

The bar root's reach in the vertical plane (the bar level, and pointing 45° down), the static joint torques with the
boom's own masses and the cutting load (`cutting.head_loads`: the feed into the wood, the chain's pull, the saw's
weight) against the drives. The boom and stick cylinders act on lever arms that change with the pose; the yaw and
the wrist are harmonic modules.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, elev, title in ((ax[0], 0.0, "bar level"), (ax[1], -math.pi / 4, "bar 45° down")):
    X, Z, ok = pbr.workspace(elevation=elev)
    a.contourf(X, Z, ok.astype(float), levels=[0.5, 1.5], colors=["#9fc5e8"])
    a.add_patch(plt.Rectangle((-Lh / 2e3, z0 / 1e3), Lh / 1e3, Hh / 1e3, color="#d9a21b", alpha=0.6))
    a.axhline(0, color="#8d7b5e"); a.set(aspect="equal", xlabel="x [m]", ylabel="z [m]", title=f"bar-root workspace, {title}")
plt.show()

In [ ]:
load = V.cut_loads()
display(pd.Series({k: v for k, v in load.items() if k.endswith("_N")}, name="head loads [N]").round(1).to_frame())
reach_rows = []
for b_ in np.linspace(0.0, 80.0, 17):
    for s_ in (-20.0, 0.0, 20.0):
        q = pbr.elevations_to_q(b_, s_, 0.0)
        root, _ = pbr.arm_fk(q)
        for with_load in (False, True):
            t = pbr.arm_torques(q, load=load if with_load else None)
            for j, dct in t.items():
                reach_rows.append({"boom_deg": b_, "stick_deg": s_, "reach_m": root[0] - ag["x"], "load": with_load, "joint": j,
                                   "tau_Nm": abs(dct["tau_Nm"]), "stall_Nm": dct["stall_Nm"], "rated_Nm": dct["rated_Nm"]})
torq = pd.DataFrame(reach_rows)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for a, j in zip(axes, ("arm_shoulder", "arm_elbow", "arm_wrist")):
    for with_load, ls in ((False, "--"), (True, "-")):
        s = torq[(torq.joint == j) & (torq.load == with_load) & (torq.stick_deg == 0.0)].sort_values("reach_m")
        a.plot(s.reach_m, s.tau_Nm, ls, label="with the cut" if with_load else "own weight")
        if with_load:
            a.plot(s.reach_m, s.rated_Nm, ":", color="k", label="drive rated")
            a.plot(s.reach_m, s.stall_Nm, "-", color="#a00", lw=0.8, label="drive stall")
    a.set(title=j, xlabel="bar root ahead of the shoulder [m]", ylabel="|τ| [N·m]"); a.legend(fontsize=8)
plt.show()
worst = torq[torq.load].assign(sf=lambda d: d.stall_Nm / d.tau_Nm).groupby("joint").sf.min()
display(worst.round(2).to_frame("lowest stall / torque with the cut"))

In [ ]:
booms = np.linspace(0, 110, 23)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(booms, [PerseusRover.lift_lever(p, b_) for b_ in booms]); ax[0].set(xlabel="boom elevation [deg]", ylabel="lever [mm]", title="lift cylinder lever about the shoulder")
els = np.linspace(*V.joint_ranges["elbow"], 26)
ax[1].plot(els, [PerseusRover.stick_lever(p, 45.0, 45.0 - e) for e in els]); ax[1].set(xlabel="stick folded on the boom, q_elbow [deg]", ylabel="lever [mm]", title="stick cylinder lever about the elbow")
plt.show()
display(pd.DataFrame({j: pbr.joint_limits(j) for j in ("yaw", "shoulder", "elbow", "wrist")}).T)

## 4. Cutting

A chainsaw removes the kerf across the branch; the energy is the wood's specific cutting energy e_c times the kerf
volume — **the one number to measure first** (`cutting.WOODS`: 35 MJ/m³ for green softwood, from a 2 kW saw
cross-cutting a 200 mm pine in ~5 s). From it: the cut time, the chain's pull, the feed and the head loads; whether
the kerf opens (cut from the top of an overhanging branch) or pinches the bar (cut from below); energy per cut.

In [ ]:
diam = [0.03, 0.05, 0.08, 0.10, 0.12, 0.15]
ct = cutting.cut_table(diam, [SAW], cutting.WOODS)
display(ct.pivot(index="d_mm", columns="wood", values="cut_time_s").round(2))
display(ct.pivot(index="d_mm", columns="wood", values="energy_Wh").round(3))
pinch = {}
for cut_from in ("top", "bottom"):
    for d in (0.06, 0.10, 0.12):
        r_ = cutting.kerf_opens(d, 1.5, V.wood, cut_from)
        pinch[(cut_from, d * 1e3)] = {"moment [N·m]": r_["moment_Nm"], "stress / MOR": r_["stress_over_MOR"], "verdict": r_["verdict"]}
display(pd.DataFrame(pinch).T)
cut_rows = pd.Series({"chain force [N]": cutting.chain_force(SAW), "feed force [N]": cutting.feed_force(SAW, None, V.k_feed),
                      "largest branch, one pass [mm]": cutting.max_branch_for_bar(SAW) * 1e3,
                      "largest branch, two cuts [mm]": cutting.max_branch_for_bar(SAW, 2) * 1e3,
                      "undercut for a 100 mm branch [mm]": cutting.undercut_depth(0.10) * 1e3,
                      "cuts of 80 mm per charge (2 s idle each)": float(cutting.cuts_per_charge(V.battery_Wh, 0.08, SAW, V.wood, overhead_Wh=2.0))},
                     name="the saw on the boom")
display(cut_rows.round(1).to_frame())

## 5. Stability with the boom out

Where the ground reaction must act for equilibrium with the boom in a pose and the cutting load on the bar: inside
the tracks' footprint the machine stands, outside it tips. The margins over the boom's elevations and the
road-wheel loads in the worst pose.

In [ ]:
grid_b, grid_s = np.linspace(0, 90, 19), np.linspace(-60, 30, 19)
M = np.full((len(grid_s), len(grid_b)), np.nan)
for i, s_ in enumerate(grid_s):
    for k, b_ in enumerate(grid_b):
        q = pbr.elevations_to_q(b_, s_, 0.0)
        if pbr.arm_fk(q)[0][2] + ag["z_hull"] < 0.05:
            continue                                       # the bar in the ground
        M[i, k] = pbr.tip_over(q, load=load)["margin_front_m"]
fig, ax = plt.subplots(figsize=(8, 5))
cs = ax.contourf(grid_b, grid_s, M * 1e3, levels=20, cmap="RdYlGn"); fig.colorbar(cs, label="front tip margin [mm]")
ax.contour(grid_b, grid_s, M, levels=[0.0], colors="k")
ax.set(xlabel="boom elevation [deg]", ylabel="stick elevation [deg]", title="front tip-over margin with the cutting load"); plt.show()
i, k = np.unravel_index(np.nanargmin(M), M.shape)
q_worst = pbr.elevations_to_q(grid_b[k], grid_s[i], 0.0)
tl = pbr.track_loads(q_worst, load=load)
print(f"worst pose: boom {grid_b[k]:.0f}°, stick {grid_s[i]:.0f}°, front margin {M[i, k] * 1e3:.0f} mm")
display(pd.DataFrame({"left [N]": tl["left_wheels_N"], "right [N]": tl["right_wheels_N"]}, index=[f"wheel {j + 1} (rear first)" for j in range(len(g['wheel_x']))]).round(0))
c = pbr.cg()
stab = tracks.stability(x_cg=c["x"], z_cg=c["z"], x_front=g["x_front"], x_rear=g["x_rear"], half_gauge=g["B"] / 2,
                        half_width=g["B"] / 2 + g["b"] / 2, step_height=g["idler"][1], contact_front=g["L"] / 2, contact_rear=-g["L"] / 2)
display(pd.Series(stab, name="boom stowed").round(2).to_frame())

## 6. The boom tube (Talos)

The boom (Al 6082-T6, 80 × 120 mm, 5 mm wall) fixed at the shoulder bore, loaded at the elbow bore with everything
outboard of it — the stick, the wrist drive, the saw and its mount — and the cutting load, twice (a snag, the
chain biting). A hand bending check alongside; then the boom's modes with the outboard mass against the shoulder
servo's own mode.

In [ ]:
AL6082 = talos.Material("Al 6082-T6 tube", youngs_modulus=70000.0, poissons_ratio=0.33, density=2.7e-9, yield_strength=260.0, source="EN 755-2")
Lb, bw, bd = p["boom_length"], p["boom_width"], p["boom_depth"]
rb = p["arm_boss_diameter"] / 2 + 0.5
REG = [talos.SurfacesInBox("shoulder", (-rb, -bw, -rb, rb, bw, rb)), talos.SurfacesInBox("elbow", (Lb - rb, -bw, -rb, Lb + rb, bw, rb))]
out_kg = sum(v for k, v in arm.items() if k.startswith(("stick", "wrist drive", "saw", "head mount")))
F_el = (out_kg * G + load["resultant_N"]) * 2
lever_out = (ag["Ls"] + ag["Lh"]) / 2 + 0.2                    # outboard centroid beyond the elbow, arm level [m]
fm = talos.StructuralModel(cad["boom"].artifacts["step"], "mm-N-MPa", AL6082, REG, [talos.FixedSupport("shoulder")],
                           [talos.Force("elbow", fz=-F_el)], talos.MeshSettings(element_size=8.0, min_element_size=2.5), name="boom_cut")
fm.mesh(RUNS / "boom_cut", progress=True); boom_res = fm.solve(RUNS / "boom_cut")
I_t = (bw * bd ** 3 - (bw - 2 * p["arm_wall"]) * (bd - 2 * p["arm_wall"]) ** 3) / 12
sig_hand = F_el * (Lb + lever_out * 1e3) * (bd / 2) / I_t
fea_tab = pd.DataFrame({"boom, level, head loads × 2": {"load at the elbow [N]": F_el, "outboard moment arm [m]": lever_out,
                         "FEA max von Mises [MPa]": boom_res.metrics.get("max_von_mises"),
                         "hand, tube bending with the outboard moment [MPa]": sig_hand,
                         "FEA max displacement [mm]": boom_res.metrics.get("max_displacement"),
                         "SF yield, hand": 260.0 / sig_hand}}).T
display(fea_tab)
mm_ = talos.StructuralModel(cad["boom"].artifacts["step"], "mm-N-MPa", AL6082, REG, [talos.FixedSupport("shoulder")], [],
                            talos.MeshSettings(element_size=10.0, min_element_size=3.0), name="boom_modes",
                            masses=[talos.PointMass("elbow", out_kg * 1e-3)])
mm_.mesh(RUNS / "boom_modes"); modes = mm_.solve_modes(RUNS / "boom_modes", n_modes=4)
kp_sh = V.arm_gains["shoulder"][0]
I_sh = out_kg * (ag["Lb"] + lever_out) ** 2 + arm.filter(like="boom (").sum() * ag["Lb"] ** 2 / 3 + V.arm_gains["shoulder"][2]
f_servo = math.sqrt(kp_sh / I_sh) / (2 * math.pi)
print(f"boom modes {[round(f, 1) for f in modes.metrics['frequencies_hz']]} Hz; the shoulder servo's own mode "
      f"(kp {kp_sh:.0f} N·m/rad on {I_sh:.0f} kg·m²) {f_servo:.1f} Hz; the chain's tooth rate is ~{SAW.chain_speed / 0.0095:.0f} Hz")

## 7. MuJoCo: the branch job

The machine in MuJoCo through Chiron (`perseus_robot.perseus()`): the tracks as Pekari's rows of velocity-servo
rollers on bogies, the boom as hinge servos (the cylinders' force on their smallest lever over the joint's range;
the brakes hold). The ground (`perseus_scenario.terrain`) is rough soil (RMS 30 mm) with an 8° side bank, two
hummocks and an even patch where the machine parks. The tree is a fixed trunk; each branch is a free capsule of
green softwood welded to it.

The scene hook `SawCutter` reads the bar's contact force on each branch every 2 ms. While the force is in the feed
window (0.15–2.5 × the saw's feed force) the cut area grows at the saw's rate, P_cut / (e_c · kerf). When it
reaches the branch's cross-section the weld is released and the branch falls. Above 4 × the feed force the bar is
jammed. The controller (`perseus_controller.branch_job`) drives to a stand-off it computes from the reach, brakes
the tracks, then for each branch: retract, swing, reach over the branch with the bar level, move in, feed under a
force loop on the cutter's reading, back off. It finishes by stowing and backing out.

In [ ]:
from vegeta import chiron
from vegeta.chiron import viz as cviz
import perseus_controller as pcc
import perseus_scenario as pss

robot = pbr.perseus()
print(f"{robot.name}: {robot.total_mass():.1f} kg, {len(robot.links())} links, {len(robot.joints())} joints "
      f"({len(robot.actuated_joints())} driven), {len(robot.feet)} rollers as feet")
lab_flat = pbr.perseus_lab(chiron.Flat(), robot=robot)
mission = pcc.CutMission([], name="stand")
ep_stand = lab_flat.run(mission, duration=1.0, rules=None, settle=0.5)
fz = np.asarray(ep_stand.log["foot_force"])[-1, :, 2]
print(f"standing: the roller loads sum to {fz.sum():.0f} N vs the weight {robot.total_mass() * G:.0f} N")

In [ ]:
scene = pss.Scene()
print(f"stand-off from the trunk {pcc.stand_distance(scene, robot):.2f} m; branches: " +
      ", ".join(f"{b.name} Ø{b.d * 1e3:.0f} mm at {b.z:.2f} m ({b.mass_kg:.1f} kg)" for b in scene.branches))
lab = pss.make_lab(scene, robot=robot, log_geoms=True)
ep = pss.run(lab, scene, duration=110.0)
ep.save(RUNS / "perseus_branches.npz")
for t_ev, kind, detail in ep.log["events"]:
    print(f"  t = {t_ev:6.2f} s  {kind}: {detail}")
print("mission finished:", ep.log["mission_finished"], "| cut:", {k: round(v, 1) for k, v in ep.log["cut_at"].items()})
ts = pss.timeseries(ep)
phases = pss.phase_table(ep, robot)
display(phases.style.format("{:.2f}"))

In [ ]:
h = ep.log["cutter_history"]
names = ep.log["branches"]
fig, ax = plt.subplots(2, 2, figsize=(15, 8))
for k, b in enumerate(names):
    ax[0, 0].plot(h[:, 0], h[:, 1 + 2 * k], label=b)
    full = cutting.cut_area(scene.branches[k].d)
    ax[0, 1].plot(h[:, 0], h[:, 2 + 2 * k] / full * 100, label=b)
for y, lab_ in ((lab.cutter.F_min, "feed window"), (lab.cutter.F_max, None), (lab.cutter.F_jam, "jam")):
    ax[0, 0].axhline(y, color="k", ls=":", lw=0.8)
ax[0, 0].set(xlabel="t [s]", ylabel="bar force (filtered) [N]", title="bar force on each branch"); ax[0, 0].legend()
ax[0, 1].set(xlabel="t [s]", ylabel="cut [% of the section]", title="the cut"); ax[0, 1].legend()
ax[1, 0].plot(ts.t, ts.x, label="x"); ax[1, 0].plot(ts.t, ts.y, label="y"); ax[1, 0].plot(ts.t, ts.tilt_deg / 10, label="tilt / 10 [deg]")
ax[1, 0].set(xlabel="t [s]", ylabel="[m]", title="hull"); ax[1, 0].legend()
for b in names:
    ax[1, 1].plot(ts.t, ts[f"{b}_z"], label=b)
ax[1, 1].set(xlabel="t [s]", ylabel="z [m]", title="branch base heights (a fall is a cut)"); ax[1, 1].legend()
plt.tight_layout(); plt.show()

In [ ]:
dt_log = float(ep.log["t"][1] - ep.log["t"][0]); every = max(1, int(round(1 / (25 * dt_log))))
imgs = cviz.frames(ep, camera="follow", every=every, size=(960, 540))
movie = cviz.to_video(imgs, OUT / "perseus_branches.mp4", fps=25)
print(f"movie: {movie} ({len(imgs)} frames)")
fig, axes = plt.subplots(1, 4, figsize=(18, 3.2))
for a, j in zip(axes, np.linspace(0, len(imgs) - 1, 4).astype(int)):
    a.imshow(imgs[j]); a.set_axis_off(); a.set_title(f"t = {j * every * dt_log:.1f} s")
plt.show()

## 8. Export

In [ ]:
for name, df in (("mass_budget", budget.to_frame()), ("arm_masses", arm.to_frame()), ("soil", soil_tab), ("drive", drive_tab),
                 ("endurance", end_tab.to_frame()), ("arm_torques", torq), ("cutting", ct), ("fea", fea_tab), ("mujoco_phases", phases)):
    df.to_csv(OUT / f"{name}.csv")
summary_doc = {"source": "37_perseus_rover", "design": {"file": "../designs/perseus_rover.py", "parameters": p},
               "mass_kg": float(budget.sum()), "boom_kg": float(arm.sum()), "ground_pressure_kPa": p_nom / 1e3, "mmp_kPa": mmp / 1e3,
               "reach_m": ag["reach"], "lowest_stall_over_torque": worst.to_dict(), "worst_front_tip_margin_m": float(np.nanmin(M)),
               "saw": SAW.as_dict(), "fea": fea_tab.to_dict(orient="index"),
               "mujoco": {"events": ep.log["events"], "cut_at": ep.log["cut_at"], "finished": ep.log["mission_finished"]},
               "debug": DEBUG}
(OUT / "perseus_rover.json").write_text(json.dumps(summary_doc, indent=2, default=float))
print("written:", sorted(x.name for x in OUT.iterdir()))

**What the numbers say** (from the check runs before this was committed; read the tables above for the run you made):

- **Mass.** About 181 kg, of which the boom assembly is 51 kg. The tracks are the heaviest group: 2 × 53 links,
  their pins and 12 road wheels weigh about 45 kg. The boom ahead of the hull centre pulls the battery to the rear
  wall. Stowed, the CG sits about 7 cm ahead of the road wheels' centre.
- **Ground.** The nominal pressure is under 10 kPa, below a person on foot. The 150 mm tracks float on soft soils.
- **Drive.** The two 1.5 kW gearmotors hold 30° within their continuous rating. On grass and gravel the grip, not
  the motors, sets the limit, as on the Pekari.
- **Boom.** The cylinders' lever arms vary from 180 to 330 mm over the boom's range, and the stall-to-torque margins
  stay above 1.5 with the cutting load. The wrist module is the tightest: 150 N·m against about 76 N·m with the saw out.
- **Cutting.** A 110 mm green softwood branch takes about 2 s of cutting at the saw's full rate. A branch cut from
  below on its overhang pinches the bar, so the job cuts from the top.
- **Stability.** The cutting load at full reach takes most of the front tip margin. Keep the boom above about 20°
  when reaching far.
- **MuJoCo.** In the check run all three branches were cut: 60 mm, 80 mm, then the 110 mm one after the bar jammed
  for about 10 s. The wrist module sits at its stall while the bar presses on the thick branch, and the force loop
  backs the bar off until it cuts again. The machine then stowed and backed out. The tracks are rows of rollers and
  the ground is rigid: read the forces as the right order, not to the newton.

**Next steps an engineer would take:**
- Measure the specific cutting energy of green branches with this saw: every cut time scales with it.
- Give the wrist a stronger module: it saturates on the 110 mm branch.
- Run the FEA without the mocks on a machine with gmsh and CalculiX.
- Check the link and pin stresses at 180 kg (the Pekari's PA6 links are scaled, not re-checked).

In [ ]:
GATE.leave()                   # nothing promised to other notebooks: the exit gate only confirms it
